# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eshaamirmalik-sketch/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row represents one pseudonymized content item for one pseudonymized client on one report date.

**Time window:** I use March 2026 as the working month for verification and feature development. The March partition covers report dates from 2026-03-01 through 2026-03-31. The warehouse overall covers 2025-01-27 through 2026-06-30. Features must use information available by the decision point; future information belongs only to the outcome/label.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Feature

I will use information that is knowable before the decision point. Candidate features include prior-period impressions, prior-period clicks, prior-period sessions, content age, and days since the last update.

### Label / proxy

The outcome proxy is future content decline: whether the page shows a meaningful decline in search impressions after the decision point. This outcome is used to rank refresh opportunities and is never used as a feature.

### Context

`client_hash_id` and `content_hash_id` are pseudonymous identifiers used for grouping, joining, and checking the grain. `report_date` defines the time window and temporal split. These fields are context, not model features.

### Excluded

I exclude `trend_pct`, `trend_direction`, and `is_declining_label` because they are derived from the decline outcome and would leak the answer into the features. I also exclude provider/model fields because they describe generation context rather than the page's refresh opportunity.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [36]:


# Query 1 — Verify grain
grain_check = con.execute("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


**Grain check result:** Observed zero duplicate combinations of `report_date`, `client_hash_id`, and `content_hash_id` in the March 2026 partition. This supports the stated grain of one content item for one client on one report date.

In [37]:
# Query 2 — March 2026 row count and date span
march_summary = con.execute("""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

march_summary

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


**March 2026 count and date-span result:** The March 2026 partition contains 9,841,378 rows, with observed dates from 2026-03-01 through 2026-03-31.

In [38]:

# Query 3 — GSC availability
availability_check = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS available_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

availability_check

,total_rows,available_rows
0,9841378,3611061


**Availability result:** In the March 2026 partition, 3,611,061 of 9,841,378 rows have `gsc_data_available IS TRUE`. This shows that GSC availability is limited in this slice, so rows without GSC data should not be treated as zero search activity.

### Five features and decision moment

For this small demonstration, the decision point is the end of 2026-03-30. I build features only from information available on or before that point and use 2026-03-31 as the outcome day.

The five features are:
1. `impressions_prev` — prior-day search impressions.
2. `clicks_prev` — prior-day search clicks.
3. `sessions_prev` — prior-day sessions.
4. `content_age_days` — content age available at the decision point.
5. `days_since_last_update` — time since the last update available at the decision point.

Each feature is used only as information available before the outcome day.

In [39]:

# Five-feature frame
features = con.execute("""
SELECT
    client_hash_id,
    content_hash_id,

    gsc_impressions AS gsc_impressions_prev,
    gsc_clicks AS gsc_clicks_prev,
    ga4_sessions AS ga4_sessions_prev,
    gsc_avg_position AS gsc_avg_position_prev,
    scroll_events AS scroll_events_prev

FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)

WHERE report_date = '2026-03-30'

LIMIT 100
""").df()

features.head()

,client_hash_id,content_hash_id,gsc_impressions_prev,gsc_clicks_prev,ga4_sessions_prev,gsc_avg_position_prev,scroll_events_prev
0,client_62f4a7e64f5e0096,content_81609787b7197d82,125,0,<NA>,1.632000,<NA>
1,client_62f4a7e64f5e0096,content_46fb7b4e297a01b0,5,0,<NA>,4.600000,<NA>
2,client_62f4a7e64f5e0096,content_04ae04b0d4ecd3bc,4,1,<NA>,4.750000,<NA>
3,client_62f4a7e64f5e0096,content_8ef30459c69a0cb4,0,0,<NA>,NaN,<NA>
4,client_62f4a7e64f5e0096,content_990a23e557819811,9,0,<NA>,17.666667,<NA>


### Five-feature availability

**`gsc_impressions_prev` — available when?** Knowable at the decision moment because it is the completed March 30 search-impression total.

**`gsc_clicks_prev` — available when?** Knowable at the decision moment because it is the completed March 30 search-click total.

**`ga4_sessions_prev` — available when?** Knowable at the decision moment because it is the completed March 30 GA4 session total, when GA4 data is available.

**`gsc_avg_position_prev` — available when?** Knowable at the decision moment because it summarizes search position observed through March 30.

**`scroll_events_prev` — available when?** Knowable at the decision moment because it is the completed March 30 scroll-event total, when GA4 data is available.

In [40]:
# Deliberate leakage experiment
# We intentionally use the current day's impressions to predict
# whether impressions are declining from the previous day.

leak_test = con.execute("""
WITH daily AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
),

paired AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,

        LAG(gsc_impressions) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
        ) AS previous_impressions

    FROM daily
)

SELECT
    *,
    CASE
        WHEN previous_impressions > 0
             AND gsc_impressions < previous_impressions
        THEN 1
        ELSE 0
    END AS is_declining_label
FROM paired
WHERE report_date >= '2026-03-02'
LIMIT 1000
""").df()

leak_test.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,previous_impressions,is_declining_label
0,2026-03-02,client_0797ff3a1fc9a6a5,content_e9ee88319837fa5a,0,0,0
1,2026-03-03,client_0797ff3a1fc9a6a5,content_e9ee88319837fa5a,0,0,0
2,2026-03-04,client_0797ff3a1fc9a6a5,content_e9ee88319837fa5a,0,0,0
3,2026-03-05,client_0797ff3a1fc9a6a5,content_e9ee88319837fa5a,0,0,0
4,2026-03-06,client_0797ff3a1fc9a6a5,content_e9ee88319837fa5a,0,0,0


In [41]:
# Deliberately leaked feature:
# This uses the same-day impressions that directly determine the label.

leak_test["leaked_feature"] = leak_test["gsc_impressions"]

In [42]:
# The leaked feature is directly involved in defining the label.
# Check how strongly it agrees with the label.

leak_test[["leaked_feature", "is_declining_label"]].corr()

,leaked_feature,is_declining_label
leaked_feature,1.000000,0.114334
is_declining_label,0.114334,1.000000


### Leakage experiment — first attempt

The deliberately added current-day impressions feature showed only a directional relationship with the decline label in this check (correlation = 0.111). This did not produce the expected near-perfect leakage signal, so I did not treat this result as evidence of severe leakage.

In [43]:
# Deliberate leakage: copy the label directly into a fake feature.
# This is intentionally invalid and is only for demonstrating leakage.

leak_test["deliberate_leak"] = leak_test["is_declining_label"]

print("Deliberate leaked feature created.")
print(leak_test[["deliberate_leak", "is_declining_label"]].head())

Deliberate leaked feature created.
   deliberate_leak  is_declining_label
0                0                   0
1                0                   0
2                0                   0
3                0                   0
4                0                   0


In [44]:
# Deliberate leakage score
# Because the fake feature is literally the label, its accuracy is 100%.

leak_accuracy = (
    leak_test["deliberate_leak"] == leak_test["is_declining_label"]
).mean()

print(f"Leaked-feature accuracy: {leak_accuracy:.3f}")

Leaked-feature accuracy: 1.000


In [45]:
# Remove the deliberately leaked column.
honest_features = leak_test.drop(columns=["deliberate_leak"])

print("deliberate_leak removed.")
print("Honest feature columns:")
print(honest_features.columns.tolist())

deliberate_leak removed.
Honest feature columns:
['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'previous_impressions', 'is_declining_label', 'leaked_feature']


In [46]:
# Remove all deliberately leaked columns and keep the label separate.

honest_features = leak_test.drop(
    columns=["deliberate_leak", "leaked_feature", "is_declining_label"]
)

label = leak_test["is_declining_label"]

print("Final honest feature columns:")
print(honest_features.columns.tolist())

Final honest feature columns:
['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'previous_impressions']


### Leakage removed

The deliberately leaked columns `deliberate_leak` and `leaked_feature` were removed. The outcome `is_declining_label` is kept separately as the label and is not treated as a feature.

The leakage experiment produced an artificially perfect accuracy of 1.000 when the label was copied directly into a feature. This confirms that using label-derived information would make the evaluation invalid.

The final working data keeps the label separate from the available-at-decision information.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

- **Unbalanced history:** Client histories start at different times, so March 2026 coverage is not equally representative across clients.
- **GSC-only periods:** Some rows have `gsc_data_available = FALSE`, so missing GSC activity must not be interpreted as zero search performance.
- **Window overlap:** Some warehouse fields use overlapping time windows. Features must only use information that would have been available before the prediction moment.
- **Decision-support limit:** This data can support ranking and modeling decisions, but it cannot by itself establish that a content change caused a performance change.


### Data limits

This slice has an unbalanced history: different clients have different amounts of usable search and analytics history, so March 2026 should not be treated as equally complete for every client.

GA4 availability is also uneven. Rows where `ga4_data_available` is FALSE should not be interpreted as zero engagement; the warehouse documentation says these values can be zero-filled before a client's GA4 start date.

This March slice can show observed search-performance patterns and support refresh prioritization, but it cannot establish that a content change caused a later performance change. It is decision-support, not causal evidence.

The five-feature frame is also limited because the daily performance table does not contain content metadata such as content age or update age; those would require a separate metadata join.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.